# Indian Traffic Sign Detection — YOLOv11 Nano

End-to-end pipeline: dataset preparation, training, validation, inference, and export.

**20 Classes:** Stop, No Entry, Left/Right/U-Turn, No Horn, School Zone, Pedestrian Crossing, Give Way, One Way, Roundabout, Railway Crossing, Road Work, Narrow Road, Speed Limits 20–80.

**Runtime:** Enable GPU (Runtime → Change runtime type → T4 GPU).

## 1. Environment Setup

In [1]:
# Detect if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab environment.")
    import os
    from pathlib import Path
    # Only clone if the project root doesn't exist, otherwise pull latest changes
    if not Path('/content/ML-model-trafic-sign').exists() and not Path('/content/traffic_sign_detection').exists():
        !git clone https://github.com/AdityaMittha/ML-model-trafic-sign.git
    else:
        print("Project directory exists. Pulling latest code...")
        current_dir = os.getcwd()
        target_dir = '/content/ML-model-trafic-sign' if Path('/content/ML-model-trafic-sign').exists() else '/content/traffic_sign_detection'
        os.chdir(target_dir)
        !git pull
        os.chdir(current_dir)
    
    PROJECT_ROOT = Path('/content/ML-model-trafic-sign')
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path('/content/traffic_sign_detection')
else:
    print("Running locally.")
    from pathlib import Path
    # Resolve the project root relative to the notebooks/ folder
    if Path('../src').exists() or Path('../scripts').exists():
        PROJECT_ROOT = Path('..').resolve()
    else:
        PROJECT_ROOT = Path('.').resolve()

import os, sys
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Working directory: {PROJECT_ROOT}')

Running in Google Colab environment.
Working directory: /content/ML-model-trafic-sign


In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q -r requirements.txt

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 92.4 MB/s eta 0:00:00:00:0100:01
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Dataset Download

Upload datasets manually or use Kaggle API:
1. Place Indian/Mapillary datasets in `data/raw/`
2. Or run Kaggle download below (requires `kaggle.json` or `access_token` in ~/.kaggle/, or `KAGGLE_API_TOKEN` Colab Secret)

In [3]:
# Show manual download instructions
!python scripts/download_datasets.py --instructions


=== Manual Dataset Download Instructions ===

1. GTSRB (Classification - supplementary):
   URL: https://benchmark.ini.rub.de/gtsrb_dataset.html
   Place in: data/raw/gtsrb/Train/<class_id>/

2. Indian Traffic Sign Dataset (Primary - Detection):
   Search: "Indian Traffic Sign Dataset" on Kaggle/GitHub
   Place YOLO format in: data/raw/indian_traffic_sign/images/ and labels/

3. Mapillary Traffic Sign Dataset (Primary - Detection):
   URL: https://www.mapillary.com/dataset/trafficsign
   Register and download, place in: data/raw/mapillary/

4. Kaggle Traffic Signs Preprocessed:
   kaggle datasets download -d valentynsichkar/traffic-signs-preprocessed
   Place in: data/raw/kaggle_preprocessed/



In [4]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    from pathlib import Path
    try:
        from google.colab import userdata
        token = userdata.get('KAGGLE_API_TOKEN')
        if token:
            os.environ['KAGGLE_API_TOKEN'] = token
    except Exception:
        pass
    
    token = os.environ.get('KAGGLE_API_TOKEN', 'KGAT_827c741d37a498545c5df2235ec09d27')
    if token:
        print("Using KAGGLE_API_TOKEN from secrets/environment/code fallback.")
        kaggle_dir = Path.home() / '.kaggle'
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        with open(kaggle_dir / 'access_token', 'w') as f:
            f.write(token.strip())
        (kaggle_dir / 'access_token').chmod(0o600)
    else:
        print("KAGGLE_API_TOKEN not found. Please upload your kaggle.json:")
        from google.colab import files
        try:
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        except Exception:
            token_input = input("Or enter your Kaggle API token (KGAT_...): ")
            if token_input.strip():
                os.environ['KAGGLE_API_TOKEN'] = token_input.strip()
                kaggle_dir = Path.home() / '.kaggle'
                kaggle_dir.mkdir(parents=True, exist_ok=True)
                with open(kaggle_dir / 'access_token', 'w') as f:
                    f.write(token_input.strip())
                (kaggle_dir / 'access_token').chmod(0o600)

    # Download the Indonesia Traffic Sign Dataset YOLOv11 from Kaggle
    !python scripts/download_datasets.py --kaggle
else:
    print("Running locally.")
    from pathlib import Path
    import os
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    access_token = kaggle_dir / 'access_token'
    token = os.environ.get('KAGGLE_API_TOKEN', 'KGAT_827c741d37a498545c5df2235ec09d27')
    if token or access_token.exists() or kaggle_json.exists():
        print("Kaggle credentials detected. Downloading dataset...")
        if token and not access_token.exists() and not kaggle_json.exists():
            kaggle_dir.mkdir(parents=True, exist_ok=True)
            with open(access_token, 'w') as f:
                f.write(token.strip())
        !python scripts/download_datasets.py --kaggle
    else:
        print(f"Kaggle credentials not found at {kaggle_dir}. Please place 'kaggle.json' or 'access_token' there.")

Using KAGGLE_API_TOKEN from secrets/environment/code fallback.
/usr/bin/python3: No module named kaggle.__main__; 'kaggle' is a package and cannot be directly executed
2026-06-24 07:58:05,423 - ERROR - Kaggle download failed: Command '['/usr/bin/python3', '-m', 'kaggle', 'datasets', 'download', '-d', 'adityabayhaqie/indonesia-traffic-sign-dataset-yolov11', '-p', '/content/ML-model-trafic-sign/data/raw/indonesia_traffic_sign', '--unzip']' returned non-zero exit status 1.
2026-06-24 07:58:05,423 - INFO - Install kaggle CLI and configure ~/.kaggle/kaggle.json with API credentials.


## 3. Dataset Preparation

In [5]:
from src.dataset.merger import DatasetMerger
from src.config import DATA_DIR

# Clean, merge, translate classes and organize splits for the Indonesia Traffic Sign dataset
merger = DatasetMerger(output_dir=DATA_DIR / 'processed')
yaml_path = merger.run_full_pipeline()
print(f'Dataset YAML: {yaml_path}')


Dataset YAML: /content/ML-model-trafic-sign/data/processed/dataset.yaml


In [6]:
from src.dataset.validator import AnnotationValidator
from src.config import DATA_DIR, OUTPUTS_DIR
validator = AnnotationValidator()
for split in ['train', 'val', 'test']:
    img_dir = DATA_DIR / 'processed' / 'images' / split
    lbl_dir = DATA_DIR / 'processed' / 'labels' / split
    if img_dir.exists():
        stats = validator.verify_split(img_dir, lbl_dir)
        print(f'{split}: {stats["total_images"]} images, {stats["total_boxes"]} boxes')
        vis_path = OUTPUTS_DIR / 'reports' / f'annotations_{split}.png'
        validator.visualize_sample(img_dir, lbl_dir, vis_path, num_samples=8)

from IPython.display import Image, display
display(Image(filename=str(OUTPUTS_DIR / 'reports' / 'annotations_train.png')))

train: 0 images, 0 boxes


ValueError: Number of rows must be a positive integer, not 0

<Figure size 1600x0 with 0 Axes>

In [ ]:

import sys
import importlib

# Force unload the cached benchmark module if it exists
if 'scripts.benchmark' in sys.modules:
    del sys.modules['scripts.benchmark']
if 'scripts' in sys.modules:
    del sys.modules['scripts']

# Now import again
from scripts.benchmark import benchmark_model_variants


## 4. Model Benchmark (YOLOv8 vs YOLOv11)

In [ ]:
from scripts.benchmark import benchmark_model_variants

results = benchmark_model_variants(
    str(DATA_DIR / 'processed' / 'dataset.yaml'),
    OUTPUTS_DIR / 'benchmarks'
)

for name, data in results.items():
    if 'error' not in data:
        print(f'{name}: mAP50={data.get("mAP50","N/A")}, FPS={data.get("avg_fps","N/A")}, Size={data.get("size_mb","N/A")}MB')

## 5. Training (YOLOv11 Nano — Transfer Learning + AMP + Early Stopping)

In [ ]:
from src.training.trainer import TrafficSignTrainer

trainer = TrafficSignTrainer(
    model_weights='yolo11n.pt',
    data_yaml=str(DATA_DIR / 'processed' / 'dataset.yaml'),
    project_dir=OUTPUTS_DIR / 'training',
)

# Run full training for 150 epochs (optimized for T4 GPU on Colab)
EPOCHS = 150

results = trainer.train(
    resume=False,
    epochs=EPOCHS,
    batch=16,
    imgsz=640,
    device=0,
    patience=25,
    amp=True,
    cos_lr=True,
)
print('Best weights:', results['best_weights'])


In [ ]:
from IPython.display import Image, display
results_img = OUTPUTS_DIR / 'training' / 'traffic_sign_yolo11n' / 'results.png'
if results_img.exists():
    display(Image(filename=str(results_img)))

In [ ]:
%cd /content/ML-model-trafic-sign
!git pull


## 6. Validation & Metrics

In [ ]:
BEST_MODEL = str(OUTPUTS_DIR / 'training' / 'traffic_sign_yolo11n' / 'weights' / 'best.pt')
FALLBACK_MODEL = str(PROJECT_ROOT / 'runs' / 'detect' / 'outputs' / 'training' / 'traffic_sign_yolo11n' / 'weights' / 'best.pt')

import shutil
from pathlib import Path
if not Path(BEST_MODEL).exists() and Path(FALLBACK_MODEL).exists():
    Path(BEST_MODEL).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FALLBACK_MODEL, BEST_MODEL)
    print('Self-healed: copied best.pt from fallback path.')

from src.evaluation.metrics import MetricsEvaluator

evaluator = MetricsEvaluator(BEST_MODEL, str(DATA_DIR / 'processed' / 'dataset.yaml'))
report = evaluator.generate_report(OUTPUTS_DIR / 'evaluation')

for split in ['val_metrics', 'test_metrics']:
    m = report[split]
    print(f'\n=== {split} ===')
    print(f'  Precision: {m["precision"]:.4f}')
    print(f'  Recall:    {m["recall"]:.4f}')
    print(f'  F1:        {m["f1"]:.4f}')
    print(f'  mAP50:     {m["mAP50"]:.4f}')
    print(f'  mAP50-95:  {m["mAP50-95"]:.4f}')

## 7. Error Analysis (FP / FN / Failure Cases)

In [ ]:
from src.evaluation.error_analysis import ErrorAnalyzer

analyzer = ErrorAnalyzer(BEST_MODEL)
stats = analyzer.analyze_split(
    DATA_DIR / 'processed' / 'images' / 'test',
    DATA_DIR / 'processed' / 'labels' / 'test',
    OUTPUTS_DIR / 'testing' / 'test',
)

print(f'TP: {stats["true_positives"]}, FP: {stats["false_positives"]}, FN: {stats["false_negatives"]}')
for rec in analyzer.recommend_improvements(stats):
    print(f'  → {rec}')

In [ ]:
%cd /content/ML-model-trafic-sign
!git pull
import sys
if 'src.inference.detector' in sys.modules:
    del sys.modules['src.inference.detector']


## 8. Real-Time Inference

In [ ]:
from src.inference.detector import TrafficSignDetector

detector = TrafficSignDetector(BEST_MODEL, device=0)

# Test on image folder
test_img_dir = DATA_DIR / 'processed' / 'images' / 'test'
benchmark = detector.run_on_source(
    source=test_img_dir,
    display=False,
    save_json=True,
)
print(f'Avg FPS: {benchmark["avg_fps"]}, Latency: {benchmark["avg_latency_ms"]:.1f}ms')
print(f'Detections: {benchmark["total_detections"]}')

In [ ]:
# Example structured detection output
import cv2
from src.inference.output_formatter import format_detection
import json

sample_imgs = list((DATA_DIR / 'processed' / 'images' / 'test').glob('*.jpg'))[:3]
for img_path in sample_imgs:
    frame = cv2.imread(str(img_path))
    dets = detector.detect_frame(frame)
    print(f'{img_path.name}: {json.dumps(dets, indent=2)}')

In [ ]:
!pip install tensorflow tensorflow-cpu


## 9. Model Export (PyTorch, ONNX, TFLite)

In [ ]:
from src.export.exporter import ModelExporter

exporter = ModelExporter(BEST_MODEL, OUTPUTS_DIR / 'exports')
export_results = exporter.benchmark_formats(imgsz=640)

for fmt, data in export_results.items():
    if fmt == 'recommendation':
        continue
    if 'error' in data:
        print(f'{fmt}: ERROR - {data["error"]}')
    else:
        print(f'{fmt}: {data["size_mb"]}MB, {data["avg_inference_ms"]}ms, {data["avg_fps"]} FPS')

print('\nPi recommendation:', export_results.get('recommendation', {}).get('recommended_format'))

## 10. Download Trained Model

In [ ]:
import shutil
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = Path('/content/ML-model-trafic-sign') if IN_COLAB else Path('.').resolve()
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

# Ensure OUTPUTS_DIR exists before archiving
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

zip_path = '/content/traffic_sign_model_bundle.zip' if IN_COLAB else str(PROJECT_ROOT / 'traffic_sign_model_bundle.zip')

print("Creating ZIP archive...")
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', root_dir=str(OUTPUTS_DIR))

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print(f"Model bundle archived successfully at: {zip_path}")